In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm

In [ ]:
def calculate_d1_d2(S, K, T, r, sigma):

    d1 = (np.log(S/K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)

    return d1,d2

In [ ]:
def black_scholes_price(S, K, T, r, sigma):  

    d1, d2 = calculate_d1_d2(S, K, T, r, sigma)

    C = S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2)
    P = K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)

    return C,P

In [ ]:
def black_scholes_greeks(S, K, T, r, sigma):

    d1,d2 = calculate_d1_d2(S, K, T, r, sigma)

    delta_c = norm.cdf(d1)
    delta_p = -(norm.cdf(-d1))

    gamma = norm.pdf(d1) / (S * sigma * np.sqrt(T))

    vega = S * norm.pdf(d1) * np.sqrt(T)

    theta_c = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) - r * K * np.exp(-r*T) * norm.cdf(d2)
    theta_p = - (S * norm.pdf(d1) * sigma) / (2 * np.sqrt(T)) + r * K * np.exp(-r*T) * norm.cdf(-d2)

    call_greeks = {
        "delta": delta_c,
        "gamma": gamma,
        "vega": vega,
        "theta": theta_c
        }

    put_greeks = {
        "delta": delta_p,
        "gamma": gamma,
        "vega": vega,
        "theta": theta_p
    }

    return call_greeks, put_greeks

In [ ]:
S0 = 100
sigma = 0.20
mu = 0.05
r = 0.05
T = 1
K = 100
N = 5
n = 100
t_array = np.linspace(0, T, n+1)
brownian_array = [0]
price_path = [S0]
rng = np.random.default_rng(4)

delta_t = T/n

# generate stock prices at discrete intervals

for i in range(1, n+1):

    brownian_array.append(brownian_array[-1] + rng.normal(0, np.sqrt(delta_t)))
    price = S0 * np.exp((mu - 0.5 * (sigma ** 2)) * t_array[i] + sigma * brownian_array[-1])
    price_path.append(price)

In [ ]:
# generates 10 (N) random paths

randoms = np.random.normal(0, np.sqrt(delta_t), size=(N, n))
zeros = np.array([[0]] * N)

brownian_increments = np.concatenate((zeros, randoms), axis = 1)
brownian = np.cumsum(brownian_increments, axis = 1)

diffusion = np.multiply(sigma, brownian)
drift = np.multiply(t_array, (mu - 0.5 * (sigma**2)))

price_factor = np.exp(drift + diffusion)
prices = np.multiply(S0, price_factor)

transposed_prices = prices.T

In [ ]:
# we have to consider time to maturity. At maturity, t[i] = T = 1, so T - t[i] = 0
# we must exclude the last value of t
# the first value delta which occurs at original stock price must be calculated using t = 0
# the last value (value 101) does not get a delta.

In [ ]:
# consider path1

path1 = prices[0]
delta = []

for i in range(len(path1) - 1):

    tau = T - t_array[i]
    d1, _ = calculate_d1_d2(path1[i], K, tau, r, sigma)

    delta.append(norm.cdf(d1))

option_premium, _ = black_scholes_price(S0, K, T, r, sigma)

cash = option_premium - (S0 * delta[0])
cash *= np.exp(r * delta_t)

for i in range(1, len(delta)):

    change = delta[i] - delta[i-1]

    cash -= path1[i] * change

    cash *= np.exp(r * delta_t)

value = cash + path1[-1] * delta[-1]

call_payoff = max(path1[-1] - K, 0)

error = value - call_payoff
print(error)

[np.float64(0.6368306511756191), np.float64(0.6008434160714715), np.float64(0.6808412623554234), np.float64(0.6823645651732624), np.float64(0.6553605192487063), np.float64(0.6464262877724308), np.float64(0.6778339870725587), np.float64(0.6377687077647232), np.float64(0.6596376001580988), np.float64(0.6181140888348146), np.float64(0.6315356761789751), np.float64(0.586502565647812), np.float64(0.5944738640310496), np.float64(0.6924530555125132), np.float64(0.6530286909760435), np.float64(0.6916588572819573), np.float64(0.7313460010703627), np.float64(0.7093688534109206), np.float64(0.6961727014553433), np.float64(0.7039106383605157), np.float64(0.7347573315713827), np.float64(0.7853028519619532), np.float64(0.763569097079208), np.float64(0.7791079251610222), np.float64(0.7651873321355611), np.float64(0.7989646546758686), np.float64(0.8103798881117552), np.float64(0.8033801813970265), np.float64(0.8229451487380645), np.float64(0.8737206526894052), np.float64(0.8801480814649838), np.float6

In [ ]:
# Now consider across multiple pathways